# Hybrid Object-Pixel based Classification Assembly

## 1. Setup the path for the source code

In [1]:
#path setup
#since the notebook is in a subfolder, we need to add the src folder to the path
#The issue can be solved by installing the package in editable mode
from pathlib import Path
import sys
project_root = next(parent
    for parent in [Path.cwd(), *Path.cwd().parents]
    if (parent / "src" / "ML_LC_Classifier").is_dir())
src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))
from ML_LC_Classifier import mosaic_rasters

## 1.1 Perform Mosaicking (If necessary)
Mosaic land cover probability layer and segments

In [3]:
from ML_LC_Classifier import mosaic_rasters
prob_2025_path = r"C:\AFLICM_Test\hybrid_lc\2025_prob\*.tif"
prob_2020_path = r"C:\AFLICM_Test\hybrid_lc\2020_prob\*.tif"
mosaic_prob_2025 = mosaic_rasters(prob_2025_path, 
                                  output_path=r"C:\AFLICM_Test\hybrid_lc\TL_Prob_2025_mosaic.tif",
                                  method='first',
                                  nodata=0)
mosaic_prob_2020 = mosaic_rasters(prob_2020_path, 
                                  output_path=r"C:\AFLICM_Test\hybrid_lc\TL_Prob_2020_mosaic.tif",
                                  method='first',
                                  nodata=0)


In [4]:
segment_path = r"C:\AFLICM_Test\hybrid_lc\SNIC_segments\*.tif"
segments_mosaic = mosaic_rasters(segment_path, 
                                 output_path=r"C:\AFLICM_Test\hybrid_lc\segments_mosaic_v1.tif",
                                 method="first",
                                 nodata = -1)

## 2. Execute Classification Assembly
### 2025 land cover

In [6]:
from ML_LC_Classifier.hybrid_assembly import process_tile
#Setup the input and output
lc_probability_path = r"C:\AFLICM_Test\hybrid_lc\TL_Prob_2025_mosaic.tif"
segments = r"C:\AFLICM_Test\hybrid_lc\segments_mosaic.tif"
output_dir = Path(r"C:\AFLICM_Test\hybrid_lc\hybrid_outputs")
#Define custom class weights (Scale: 0.1 to 5.0)
#Higher = trust segment more (smoother); Lower = trust pixel more (sharper)
smoothing_param= {
    'Primary_forest':        4.8,
    'Secondary_forest':      4,
    'Coastal_forest':        4,
    'Mangrove_forest':       3.5,
    'Teak_plantation':       2.5,
    'Eucalyptus_plantation': 2.5,
    'Coffe_agroforestry':    2.8,
    'Cocoa_agroforestry':    2.0,
    'Coconut_agroforestry':  2.0,
    'Mixed_garden':          1.5,
    'Paddy_field':           1.5,
    'Other_cropland':        1.2,
    'Grassland':             1.8,
    'Shrubland':             1.8,
    'Cleared_land':          1.0,
    'Built-up':              1,
    'Waterbody':             4,
    'Dry_riverbed':          0.1
}
#run the entire processing chain
process_tile(lc_probability_path, segments, output_dir,
            river_corridor_path= r"C:\AFLICM_Test\hybrid_lc\mask_layer\TL_River_Corridor_Mask_30m.tif",
            settlement_path = r"C:\AFLICM_Test\hybrid_lc\mask_layer\HRSL_Settlement_Binary_UTM.tif",
            tau=0.65,
            adaptive_sensitivity=0.65,
            smoothness_dict=smoothing_param
            )


--- Processing: TL_Prob_2025_mosaic ---
Probabilities: C:\AFLICM_Test\hybrid_lc\TL_Prob_2025_mosaic.tif
Segments:      C:\AFLICM_Test\hybrid_lc\segments_mosaic.tif
Block size:    2048x2048 (overlap: 64px)
Stability tau: 0.65 | Adaptive sensitivity: 0.65
Active Mask-Layer Rules: 2
  1. False Built-up (16) in River Corridor -> Water (17)
  2. False Water ((17, 18)) in Confirmed Settlement -> Built-up (16)
Total windowed blocks: 171 (9 rows x 19 cols)
  Block  10/171 done (  5.8%)...
  Block  30/171 done ( 17.5%)...
  Block  50/171 done ( 29.2%)...
  Block  70/171 done ( 40.9%)...
  Block  90/171 done ( 52.6%)...
  Block 100/171 done ( 58.5%)...
  Block 110/171 done ( 64.3%)...
  Block 120/171 done ( 70.2%)...
  Block 140/171 done ( 81.9%)...

--- Processing Complete! ---
Outputs written to: C:\AFLICM_Test\hybrid_lc\hybrid_outputs
  1. LULC Map:        C:\AFLICM_Test\hybrid_lc\hybrid_outputs\TL_Prob_2025_mosaic_LULC.tif
  2. Stability Map:   C:\AFLICM_Test\hybrid_lc\hybrid_outputs\TL_Pro

{'lulc': WindowsPath('C:/AFLICM_Test/hybrid_lc/hybrid_outputs/TL_Prob_2025_mosaic_LULC.tif'),
 'stability': WindowsPath('C:/AFLICM_Test/hybrid_lc/hybrid_outputs/TL_Prob_2025_mosaic_Stability_Map.tif'),
 'ci': WindowsPath('C:/AFLICM_Test/hybrid_lc/hybrid_outputs/TL_Prob_2025_mosaic_Confusion_Index.tif')}

### 2020 Land Cover

In [7]:
#Setup the input and output
lc_probability_path = r"C:\AFLICM_Test\hybrid_lc\TL_Prob_2020_mosaic.tif"
segments = r"C:\AFLICM_Test\hybrid_lc\segments_mosaic.tif"
output_dir = Path(r"C:\AFLICM_Test\hybrid_lc\hybrid_outputs")
#Define custom class weights (Scale: 0.1 to 5.0)
#Higher = trust segment more (smoother); Lower = trust pixel more (sharper)
smoothness_2020 = {
    # Dense forest: moderate smoothing to remove cloud/canopy shadow noise (w = 0.44)
    'Primary_forest':        2.5,   # 44% object, 56% pixel
    'Secondary_forest':      2.2,   # 40% object, 60% pixel
    'Coastal_forest':        2.0,   # 40% object, 60% pixel
    'Mangrove_forest':       2.0,   # 40% object, 60% pixel
    'Waterbody':             3.0,   # 60% object (water bodies are naturally homogeneous)

    # Semi-dense & Agroforestry: high pixel trust (w = 0.30)
    'Teak_plantation':       1.5,   # 30% object, 70% pixel
    'Eucalyptus_plantation': 1.5,   # 30% object, 70% pixel
    'Coffe_agroforestry':    1.5,   # 30% object, 70% pixel
    'Cocoa_agroforestry':    1.5,   # 30% object, 70% pixel
    'Coconut_agroforestry':  1.5,   # 30% object, 70% pixel

    # Seasonal vegetation & Cropland: high pixel trust (w = 0.24)
    'Grassland':             1.2,   # 24% object, 76% pixel
    'Shrubland':             1.2,   # 24% object, 76% pixel
    'Mixed_garden':          1.2,   # 24% object, 76% pixel
    'Paddy_field':           1.0,   # 20% object, 80% pixel
    'Other_cropland':        1.0,   # 20% object, 80% pixel

    # Sharp boundary / human features: maximum pixel trust (w = 0.20)
    'Built-up':              0.5,   # 20% object, 80% pixel
    'Cleared_land':          0.5,   # 20% object, 80% pixel
    'Dry_riverbed':          0.1,   # 20% object, 80% pixel
}
#run the entire processing chain
process_tile(lc_probability_path, segments, output_dir,
            river_corridor_path= r"C:\AFLICM_Test\hybrid_lc\mask_layer\TL_River_Corridor_Mask_30m.tif",
            settlement_path = r"C:\AFLICM_Test\hybrid_lc\mask_layer\HRSL_Settlement_Binary_UTM.tif",
            tau=0.65,
            adaptive_sensitivity=0.65,
            smoothness_dict=smoothing_param
            )


--- Processing: TL_Prob_2020_mosaic ---
Probabilities: C:\AFLICM_Test\hybrid_lc\TL_Prob_2020_mosaic.tif
Segments:      C:\AFLICM_Test\hybrid_lc\segments_mosaic.tif
Block size:    2048x2048 (overlap: 64px)
Stability tau: 0.65 | Adaptive sensitivity: 0.65
Active Mask-Layer Rules: 2
  1. False Built-up (16) in River Corridor -> Water (17)
  2. False Water ((17, 18)) in Confirmed Settlement -> Built-up (16)
Total windowed blocks: 171 (9 rows x 19 cols)
  Block  10/171 done (  5.8%)...
  Block  30/171 done ( 17.5%)...
  Block  50/171 done ( 29.2%)...
  Block  70/171 done ( 40.9%)...
  Block  90/171 done ( 52.6%)...
  Block 100/171 done ( 58.5%)...
  Block 110/171 done ( 64.3%)...
  Block 120/171 done ( 70.2%)...
  Block 140/171 done ( 81.9%)...

--- Processing Complete! ---
Outputs written to: C:\AFLICM_Test\hybrid_lc\hybrid_outputs
  1. LULC Map:        C:\AFLICM_Test\hybrid_lc\hybrid_outputs\TL_Prob_2020_mosaic_LULC.tif
  2. Stability Map:   C:\AFLICM_Test\hybrid_lc\hybrid_outputs\TL_Pro

{'lulc': WindowsPath('C:/AFLICM_Test/hybrid_lc/hybrid_outputs/TL_Prob_2020_mosaic_LULC.tif'),
 'stability': WindowsPath('C:/AFLICM_Test/hybrid_lc/hybrid_outputs/TL_Prob_2020_mosaic_Stability_Map.tif'),
 'ci': WindowsPath('C:/AFLICM_Test/hybrid_lc/hybrid_outputs/TL_Prob_2020_mosaic_Confusion_Index.tif')}

## 3. Minimum Mapping Unit Classification Smoothing

In [ ]:
from pathlib import Path
from ML_LC_Classifier import apply_mmu

# Paths
lc_2020 = Path("C:/AFLICM_Test/hybrid_lc/hybrid_outputs/S2_2020_data/TL_Prob_2020_mosaic_LULC.tif")
lc_2025 = Path("C:/AFLICM_Test/hybrid_lc/hybrid_outputs/2025 data/TL_Prob_2025_mosaic_LULC.tif")

mmu_2020_path = Path("C:/AFLICM_Test/hybrid_lc/hybrid_outputs/TL_2020_LULC_S2_filtered.tif")
mmu_2025_path = Path("C:/AFLICM_Test/hybrid_lc/hybrid_outputs/TL_2025_LULC_S2_filtered.tif")
#the minimum mapping unit filter table

mmu_table = {
    1: 40,   #primary forest 
    2: 30,   #secondary forest 
    3: 20,   #mangrove forest
    4: 30,   #coastal forest 
    5: 15,   #teak plantation 
    6: 20,   #eucalyptus plantation
    7: 20,   #coffee agroforestry
    8: 20,   #cocoa agroforestry
    9: 20,   #coconut agroforestry
    10: 30,  #mixed garden
    11: 30,  #paddy field
    12: 30,  #other cropland
    13: 30,  #grassland
    14: 40,  #shurbland
    15: 30,  #cleared land
    16: 20,  #built-up
    17: 20,  #Waterbody
}

# 1. Apply MMU to 2020
print("Applying MMU filter to 2020 LULC...")
apply_mmu(
    raster_path=lc_2020,
    output_path=mmu_2020_path,
    table=mmu_table ,
    iterations=5,        # 4 iterations of 3x3 focal fill is plenty for 10-15px patches
    block_size=2048,     # Low-memory streaming window
    overlap=64           # Seam-free boundary overlap
)

# 2. Apply MMU to 2025
print("Applying MMU filter to 2025 LULC...")
apply_mmu(
    raster_path=lc_2025,
    output_path=mmu_2025_path,
    table=mmu_table ,
    iterations=5,
    block_size=2048,
    overlap=64
)
print("MMU filtering complete!")

Applying MMU filter to 2020 LULC...
Applying MMU filter to 2025 LULC...


RasterioIOError: C:\AFLICM_Test\hybrid_lc\hybrid_outputs\2025_data\TL_Prob_2025_mosaic_LULC.tif: No such file or directory

## 4. Consistancy Analysis and Transistion 

In [2]:
from pathlib import Path
from ML_LC_Classifier import (calculate_transition_matrix,apply_transition_consistency,DEFAULT_TIMOR_17_CLASSES)
lc_class = {
    1: "Primary_forest",
    2: "Secondary_forest",
    3: "Mangrove_forest",
    4: "Coastal_forest",
    5: "Teak_plantation",
    6: "Eucalyptus_plantation",
    7: "Coffe_agroforestry",
    8: "Cocoa_agroforestry",
    9: "Coconut_agroforestry",
    10: "Mixed_garden",
    11: "Paddy_field",
    12: "Other_cropland",
    13: "Grassland",
    14: "Shrubland",
    15: "Cleared_land",
    16: "Built-up",
    17: "Waterbody",
}
# 1. Inputs: MMU-filtered 2020 and 2025 rasters
t1_2020 = Path(r"C:\AFLICM_Test\hybrid_lc\hybrid_outputs\MMU_filtered_output\TL_2020_LULC_S2_filtered.tif")
t2_2025 = Path("C:\AFLICM_Test\hybrid_lc\hybrid_outputs\MMU_filtered_output\TL_2025_LULC_S2_filtered.tif")
# 2. Step 1: Cross-tabulation (Identify existing transitions & area in ha)
print("Computing transition cross-tabulation matrix...")
df_counts, df_areas = calculate_transition_matrix(
    t1_path=t1_2020,
    t2_path=t2_2025,
    class_names=DEFAULT_TIMOR_17_CLASSES,  # Or pass any custom dict: {1: "Forest", ...}
    csv_path="C:/AFLICM_Test/hybrid_lc/hybrid_outputs/transition_matrix_2020_2025.csv"
)
# Display top transitions or examine specific pairs
print(df_areas)
# 3. Step 2: Define forbidden (impossible) transitions
# e.g., (from_class, to_class)
# In a 5-year span: Secondary Forest (2) cannot become Primary Forest (1)
# Built-up (16) cannot become Primary Forest (1) or Mangrove (3)
forbidden_rules = [
    (2, 1),   # Secondary Forest -> Primary Forest (impossible in 5 years)
    (13, 1),  # Grassland -> Primary Forest
    (14, 1),  # Shrubland -> Primary Forest
    (15, 1),  # Cleared land -> Primary Forest
    (16, 1),  # Built-up -> Primary Forest
    (16, 2),  # Built-up -> Secondary Forest
    (16, 3),  # Built-up -> Mangrove
]

# 4. Step 3: Apply consistency correction to 2025
output_2025_corrected = Path("C:/AFLICM_Test/hybrid_lc/hybrid_outputs/TL_2025_LULC_Consistent.tif")
flag_raster = Path("C:/AFLICM_Test/hybrid_lc/hybrid_outputs/TL_transition_corrections_flag.tif")
print("Applying transition consistency...")
apply_transition_consistency(
    t1_path=t1_2020,
    t2_path=t2_2025,
    output_path=output_2025_corrected,
    forbidden=forbidden_rules,
    correction_target="t2",   # Demote/revert impossible 2025 pixels back to realistic classes
    flag_path=flag_raster     # Generates a map of where corrections took place
)
print("Consistency correction completed successfully!")

Computing transition cross-tabulation matrix...
                       Primary_forest  Secondary_forest  Mangrove_forest  \
From (t1)                                                                  
Primary_forest              225411.40            870.52             9.57   
Secondary_forest             17852.90         217963.47            47.62   
Mangrove_forest                 10.03             54.61          2516.75   
Coastal_forest                 355.95              3.23           185.57   
Teak_plantation                328.77           1030.10             0.74   
Eucalyptus_plantation         1122.75            488.39             3.78   
Coffe_agroforestry           15368.93           1513.50            26.45   
Cocoa_agroforestry            5944.25          18058.50             0.12   
Coconut_agroforestry           542.55            103.24           109.77   
Mixed_garden                  1503.09           1079.99            36.49   
Paddy_field                    184.19   

## Corrected Transition Rule for 2025

In [3]:
from pathlib import Path
from ML_LC_Classifier import apply_transition_consistency

# Paths
t1_2020 = Path(r"C:\AFLICM_Test\hybrid_lc\hybrid_outputs\MMU_filtered_output\TL_2020_LULC_S2_filtered.tif")
t2_2025 = Path(r"C:\AFLICM_Test\hybrid_lc\hybrid_outputs\MMU_filtered_output\TL_2025_LULC_S2_filtered.tif")

out_2025_corrected = Path(r"C:\AFLICM_Test\hybrid_lc\hybrid_outputs\TL_2025_LULC_Consistent.tif")
flag_raster = Path(r"C:\AFLICM_Test\hybrid_lc\hybrid_outputs\TL_2025_Consistency_Flags.tif")

# 1. Define the complete list of forbidden transitions
forbidden_rules = [
    # Group A: Impossible Primary Forest gains in 5 years
    (2, 1),   # Secondary Forest -> Primary Forest (17,852 ha)
    (5, 1),   # Teak -> Primary Forest
    (6, 1),   # Eucalyptus -> Primary Forest
    (7, 1),   # Coffee -> Primary Forest (15,368 ha)
    (8, 1),   # Cocoa -> Primary Forest (5,944 ha)
    (9, 1),   # Coconut -> Primary Forest
    (10, 1),  # Mixed garden -> Primary Forest
    (11, 1),  # Paddy -> Primary Forest
    (12, 1),  # Other cropland -> Primary Forest
    (13, 1),  # Grassland -> Primary Forest
    (14, 1),  # Shrubland -> Primary Forest
    (15, 1),  # Cleared land -> Primary Forest
    
    # Group B: False Urban De-urbanization
    (16, 1),  # Built-up -> Primary Forest
    (16, 2),  # Built-up -> Secondary Forest
    (16, 3),  # Built-up -> Mangrove
    (16, 17), # Built-up -> Waterbody (urban asphalt/shadow)
    
    # Group C: False Water / Shadow Artifacts
    (1, 17),  # Primary Forest -> Waterbody (mountain shadow)
    (2, 17),  # Secondary Forest -> Waterbody
    (15, 17), # Cleared land -> Waterbody (wet bare soil)
]

# 2. Run consistency correction on 2025
print("Running consistency correction on 2025...")
apply_transition_consistency(
    t1_path=t1_2020,
    t2_path=t2_2025,
    output_path=out_2025_corrected,
    forbidden=forbidden_rules,
    correction_target="t2",   # Corrects 2025 map
    flag_path=flag_raster,     # Maps where corrections were made
    block_size=2048
)

print("2025 Consistency correction complete! Re-calculating final transition matrix...")

Running consistency correction on 2025...
2025 Consistency correction complete! Re-calculating final transition matrix...


In [ ]:
lc_2025 = Path("C:/AFLICM_Test/hybrid_lc/hybrid_outputs/2025 data/TL_Prob_2025_mosaic_LULC.tif")

print("Applying MMU filter to 2025 LULC...")
apply_mmu(
    raster_path=lc_2025,
    output_path=mmu_2025_path,
    table=mmu_table ,
    iterations=5,
    block_size=2048,
    overlap=64
)
print("MMU filtering complete!")

Applying MMU filter to 2025 LULC...
MMU filtering complete!


In [6]:
from ML_LC_Classifier import apply_mmu
lc_2025_pixel = Path(r"C:\AFLICM_Test\hybrid_lc\Pixel_based_Land Cover\LULC_Landsat_2025_pixel.tif")
lc_2020_pixel = Path(r"C:\AFLICM_Test\hybrid_lc\Pixel_based_Land Cover\LULC_Landsat_2020_pixel.tif")
mmu_2020_path = Path(r"C:\AFLICM_Test\hybrid_lc\Pixel_based_Land Cover\LULC_S2_2020_pixel_filtered.tif")
mmu_2025_path = Path(r"C:\AFLICM_Test\hybrid_lc\Pixel_based_Land Cover\LULC_S2_2025_pixel_filtered.tif")
mmu_table = {
    1: 40,   #primary forest 
    2: 30,   #secondary forest 
    3: 20,   #mangrove forest
    4: 30,   #coastal forest 
    5: 15,   #teak plantation 
    6: 20,   #eucalyptus plantation
    7: 20,   #coffee agroforestry
    8: 20,   #cocoa agroforestry
    9: 20,   #coconut agroforestry
    10: 30,  #mixed garden
    11: 30,  #paddy field
    12: 30,  #other cropland
    13: 30,  #grassland
    14: 40,  #shurbland
    15: 30,  #cleared land
    16: 20,  #built-up
    17: 20,  #Waterbody
}
print("Applying MMU filter to 2025 LULC...")

apply_mmu(
    raster_path=lc_2025_pixel,
    output_path=mmu_2025_path,
    table=mmu_table ,
    iterations=5,
    block_size=2048,
    overlap=64
)
print("Applying MMU filter to 2020 LULC...")
apply_mmu(
    raster_path=lc_2020_pixel,
    output_path=mmu_2020_path,
    table=mmu_table ,
    iterations=5,
    block_size=2048,
    overlap=64
)
print("MMU filtering complete!")

Applying MMU filter to 2025 LULC...
Applying MMU filter to 2020 LULC...
MMU filtering complete!


In [7]:
from pathlib import Path
from ML_LC_Classifier import (calculate_transition_matrix,apply_transition_consistency)
lc_class = {
    1: "Primary_forest",
    2: "Secondary_forest",
    3: "Mangrove_forest",
    4: "Coastal_forest",
    5: "Teak_plantation",
    6: "Eucalyptus_plantation",
    7: "Coffe_agroforestry",
    8: "Cocoa_agroforestry",
    9: "Coconut_agroforestry",
    10: "Mixed_garden",
    11: "Paddy_field",
    12: "Other_cropland",
    13: "Grassland",
    14: "Shrubland",
    15: "Cleared_land",
    16: "Built-up",
    17: "Waterbody",
}
# 1. Inputs: MMU-filtered 2020 and 2025 rasters
t1_2020 = Path(r"C:\AFLICM_Test\hybrid_lc\Pixel_based_Land Cover\LULC_S2_2020_pixel_filtered.tif")
t2_2025 = Path(r"C:\AFLICM_Test\hybrid_lc\Pixel_based_Land Cover\LULC_S2_2025_pixel_filtered.tif")
# 2. Step 1: Cross-tabulation (Identify existing transitions & area in ha)
print("Computing transition cross-tabulation matrix...")
df_counts, df_areas = calculate_transition_matrix(
    t1_path=t1_2020,
    t2_path=t2_2025,
    class_names=lc_class,  # Or pass any custom dict: {1: "Forest", ...}
    csv_path="C:/AFLICM_Test/hybrid_lc/hybrid_outputs/transition_matrix_2020_2025_pixel.csv"
)
# Display top transitions or examine specific pairs
print(df_areas)
# 3. Step 2: Define forbidden (impossible) transitions
# e.g., (from_class, to_class)
# In a 5-year span: Secondary Forest (2) cannot become Primary Forest (1)
# Built-up (16) cannot become Primary Forest (1) or Mangrove (3)
forbidden_rules = [
    (2, 1),   # Secondary Forest -> Primary Forest (impossible in 5 years)
    (13, 1),  # Grassland -> Primary Forest
    (14, 1),  # Shrubland -> Primary Forest
    (15, 1),  # Cleared land -> Primary Forest
    (16, 1),  # Built-up -> Primary Forest
    (16, 2),  # Built-up -> Secondary Forest
    (16, 3),  # Built-up -> Mangrove
]

# 4. Step 3: Apply consistency correction to 2025
output_2025_corrected = Path("C:/AFLICM_Test/hybrid_lc/hybrid_outputs/TL_2025_LULC_Consistent_pixel.tif")
flag_raster = Path("C:/AFLICM_Test/hybrid_lc/hybrid_outputs/TL_transition_corrections_flag_pixel.tif")
print("Applying transition consistency...")
apply_transition_consistency(
    t1_path=t1_2020,
    t2_path=t2_2025,
    output_path=output_2025_corrected,
    forbidden=forbidden_rules,
    correction_target="t2",   # Demote/revert impossible 2025 pixels back to realistic classes
    flag_path=flag_raster     # Generates a map of where corrections took place
)
print("Consistency correction completed successfully!")

Computing transition cross-tabulation matrix...
                       Primary_forest  Secondary_forest  Mangrove_forest  \
From (t1)                                                                  
Primary_forest              195975.08           1149.53            38.36   
Secondary_forest             16138.89         223359.41            95.51   
Mangrove_forest                 36.35            101.43          2894.40   
Coastal_forest                 267.07              2.12           209.27   
Teak_plantation                368.54           1762.33             2.05   
Eucalyptus_plantation         1782.99            969.43            16.21   
Coffe_agroforestry           16796.55           1920.91            70.00   
Cocoa_agroforestry            6276.70          21577.61             2.56   
Coconut_agroforestry           847.24             79.85           166.90   
Mixed_garden                  1463.57            996.47            52.77   
Paddy_field                    164.15   